In [15]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib


df = pd.read_csv("Merged file.csv")
df.columns = df.columns.str.strip()

# Convert required columns
cols = ["Age", "Gross monthly income", "Insurance", "Emergency Fund", "Savings", "Debt", "Investments", "Net monthly income"]
for col in cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ""), errors='coerce')

# Calculate HLV & label
def calculate_insurance_gap(row, retirement_age=60):
    age = row["Age"]
    income = row["Gross monthly income"]
    insurance = row["Insurance"]
    
    if pd.isnull(age) or pd.isnull(income) or pd.isnull(insurance):
        return pd.NA, pd.NA

    years_remaining = max(0, retirement_age - age)
    hlv = income * 12 * years_remaining * 0.5

    if insurance < 0.7 * hlv:
        label = "Underinsured"
    elif insurance > 1.2 * hlv:
        label = "Overinsured"
    else:
        label = "Adequately Insured"

    return hlv, label

df[["HLV", "Insurance Gap"]] = df.apply(lambda row: pd.Series(calculate_insurance_gap(row)), axis=1)
df = df.dropna(subset=["Insurance Gap"])

# Encode target
label_encoder = LabelEncoder()
df["Insurance Gap Label"] = label_encoder.fit_transform(df["Insurance Gap"])

# Define features and train
features = ["Age", "Gross monthly income", "Net monthly income", "Savings", "Debt", "Emergency Fund", "Investments"]
X = df[features]
y = df["Insurance Gap Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


# Save model
joblib.dump(model, "insurance_gap_model.pkl")
joblib.dump(label_encoder, "insurance_gap_label_encoder.pkl")


Classification Report:
               precision    recall  f1-score   support

 Overinsured       0.00      0.00      0.00         1
Underinsured       0.97      1.00      0.98        29

    accuracy                           0.97        30
   macro avg       0.48      0.50      0.49        30
weighted avg       0.93      0.97      0.95        30


Confusion Matrix:
 [[ 0  1]
 [ 0 29]]


['insurance_gap_label_encoder.pkl']

**Function: Insurance Gap + HLV + Health Risk Scoring**

In [16]:
import pandas as pd
import joblib

# Load model & label encoder
model = joblib.load("insurance_gap_model.pkl")
label_encoder = joblib.load("insurance_gap_label_encoder.pkl")

def predict_insurance_gap_with_health(user_data: dict):
    """
    Predicts insurance gap, calculates HLV, and scores health risk.
    Returns: gap label, HLV, health risk level, and advice.
    """
    import pandas as pd

    # Extract fields
    age = user_data["Age"]
    income = user_data["Gross monthly income"]
    insurance = user_data.get("Insurance", 0)
    occupation = user_data.get("Occupation", "").lower()
    lifestyle = user_data.get("Lifestyle Score", 5)
    family_history = user_data.get("Family History", "No").lower()

    # Calculate HLV
    years_remaining = max(0, 60 - age)
    hlv = income * 12 * years_remaining * 0.5

    # Insurance gap prediction
    df = pd.DataFrame([{
        k: user_data[k] for k in [
            "Age", "Gross monthly income", "Net monthly income", 
            "Savings", "Debt", "Emergency Fund", "Investments"
        ]
    }])
    
    pred_encoded = model.predict(df)[0]
    gap_label = label_encoder.inverse_transform([pred_encoded])[0]

    # Health risk scoring
    risk_score = 0
    if age > 50: risk_score += 2
    if lifestyle <= 5: risk_score += 2
    if "yes" in family_history: risk_score += 3
    if any(word in occupation for word in ["desk", "office", "it", "developer"]):
        risk_score += 1

    if risk_score <= 2:
        health_risk = "Low"
    elif risk_score <= 5:
        health_risk = "Medium"
    else:
        health_risk = "High"

    # Advice
    advice = [f" Your estimated Human Life Value (HLV) is ₹{hlv:,.0f}."]

    if gap_label == "Underinsured":
        advice.append(f" You're underinsured. You may need ₹{hlv - insurance:,.0f} more coverage.")
    elif gap_label == "Overinsured":
        advice.append(" You may be paying more than needed for life cover.")
    else:
        advice.append(" Your insurance appears adequate based on income and age.")

    if health_risk == "High":
        advice.append(" You are at high health risk. Consider critical illness and health insurance urgently.")
    elif health_risk == "Medium":
        advice.append(" You are at moderate health risk. A health plan with OPD coverage may be beneficial.")
    else:
        advice.append(" Your health risk is low. Maintain a good lifestyle and consider wellness-based plans.")

    return gap_label, hlv, health_risk, advice


**Testing example**

In [17]:
user = {
    "Age": 52,
    "Gross monthly income": 80000,
    "Net monthly income": 65000,
    "Savings": 200000,
    "Debt": 30000,
    "Emergency Fund": 50000,
    "Investments": 100000,
    "Insurance": 700000,
    "Occupation": "IT Manager",
    "Lifestyle Score": 4,
    "Family History": "Yes"
}

gap, hlv, risk, advice = predict_insurance_gap_with_health(user)

print(" Insurance Gap:", gap)
print(" HLV:", round(hlv))
print(" Health Risk:", risk)
print(" Advice:")
for line in advice:
    print("-", line)


 Insurance Gap: Underinsured
 HLV: 3840000
 Health Risk: High
 Advice:
-  Your estimated Human Life Value (HLV) is ₹3,840,000.
-  You're underinsured. You may need ₹3,140,000 more coverage.
-  You are at high health risk. Consider critical illness and health insurance urgently.
